# Run an analysis pipeline on computed embeddings

This notebook demonstrates the GELOS analysis pipeline for pre-computed Earth Observation embeddings. First, a subset of embeddings are extracted. Then, these are transformed and used for modeling.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# top level imports

from pprint import pprint
from pathlib import Path
import yaml
from gelos.extraction import extract_embeddings
from gelos.transforms import TRANSFORMS
from gelos.models import MODELS
from gelos.analysis import load_chip_tracker
from gelos.plotting import PLOTS, build_style_from_config

In [ ]:
# set up paths for this analysis run

raw_data_dir = Path("/app/data/raw")
embedding_dir = Path("/app/data/interim")
figures_dir = Path("/app/reports/figures")
yaml_path = Path("/app/configs/exp004_prithvi600.yaml")

In [ ]:
# read yaml config and construct variables for this run automatically

with open(yaml_path, "r") as f:
    yaml_config = yaml.safe_load(f)

data_version = yaml_config['data_version']
chip_tracker_file = yaml_config["chip_tracker"]
chip_id_column = yaml_config["chip_id_column"]
data_root = raw_data_dir / data_version

chip_gdf = load_chip_tracker(data_root / chip_tracker_file)
chip_gdf = chip_gdf.set_index(chip_id_column)

config_stem = yaml_path.stem
output_dir = embedding_dir / data_version / config_stem

## Select embedding layer for this demonstration

In [ ]:
# show embedding layer directories

embedding_dirs = list(output_dir.glob("*"))
for dir in embedding_dirs:
    print(str(dir))

In [ ]:
# select embedding layer based on options

embedding_layer = "layer_31"
embeddings_directory = output_dir / embedding_layer

## Select extraction strategy for this demonstration

In [ ]:
# show extraction strategies

print("Available straegies in config:\n------------")
for strategy_key, strategy_cfg in yaml_config["embedding_extraction_strategies"].items():
    print(f"{strategy_cfg['title']} -  strategy key: {strategy_key}")

In [ ]:
# select strategy key from options

strategy_key = "all_steps_of_middle_patch"

## Extract embeddings

In [ ]:
# slice embeddings according to slice args

slice_args = strategy_cfg["slice_args"]
strategy_title = strategy_cfg.get("title", strategy_key)
prefix = f"{config_stem}_{strategy_key}_{embedding_layer}"
layer_dir = output_dir / embedding_layer

# since we are using a sample, do not cache to disk
embeddings, chip_indices = extract_embeddings(
    embeddings_directory, n_sample=1000, slice_args=slice_args,
)

## Step 2: Transforms

In [ ]:
# PCA transformation

pca_result = TRANSFORMS["pca"](embeddings)

In [ ]:
# t-SNE transformation

tsne_result = TRANSFORMS["tsne"](embeddings, perplexity=30)

In [ ]:
# UMAP transformation

umap_result = TRANSFORMS["umap"](embeddings, random_state=None)

## Step 3: Plotting and Modeling

### t-SNE plot

In [ ]:
experiment_name = yaml_config["experiment_name"]
style_cfg = yaml_config["style"]
category_column, _, _ = build_style_from_config(style_cfg)

p_type = "scatter_2d"
p_params = {
    "axis_lim" : 160
}

data = umap_result
t_type = "umap"
output_path = None
p_fn = PLOTS[p_type]

p_fn(
    data,
    chip_gdf,
    chip_indices,
    style_cfg,
    experiment_name,
    strategy_title,
    t_type,
    embedding_layer,
    **p_params,
)


### Run models with raw embeddings and outputs of t-SNE

In [ ]:
# retrieve labels based on chip indices returned by embedding slicing
labels = chip_gdf[category_column].loc[chip_indices].to_numpy()

#### KNN Classifier

In [ ]:
knn_results = MODELS["knn"](embeddings, labels, output_dir=Path("/app/reports"), run_name="knn_test")
print(f"KNN Accuracy on Sliced Embeddings: {knn_results['accuracy']}")
print(f"Per Class Accuracy on Sliced Embeddings: {knn_results['per_class']}")

#### Random Forest Classifier

In [ ]:
randomforest_results = MODELS["random_forest"](embeddings, labels, output_dir=Path("/app/reports"), run_name="rf_test")
print(f"Accuracy on Sliced Embeddings: {randomforest_results['accuracy']}")
print(f"Per Class Accuracy on Sliced Embeddings: {randomforest_results['per_class']}")

In [ ]:
randomforest_results = MODELS["random_forest"](tsne_result, labels, output_dir=Path("/app/reports"), run_name="rf_test")
print(f"Accuracy on t-SNE Transformed Embeddings: {randomforest_results['accuracy']}")
print(f"Per Class Accuracy on t-SNE Transformed Embeddings: {randomforest_results['per_class']}")

#### Linear Probe Classifier

In [ ]:
linearprobe_results = MODELS["linear_probe"](tsne_result, labels, output_dir=Path("/app/reports"), run_name="linearprobe_test")
print(f"Accuracy on t-SNE Transformed Embeddings: {linearprobe_results['accuracy']}")
print(f"Per Class Accuracy on t-SNE Transformed Embeddings: {linearprobe_results['per_class']}")